In [1]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep
import os

from myutils import email_notify


from datetime import datetime
today = datetime.strftime(datetime.today(), '%Y%m%d')
del datetime

In [2]:
today

'20241105'

In [3]:
# with RaspiLED() as led:
#     led.check()

In [4]:
np.linspace(0.1, 0.4, 61)[:31]

array([0.1  , 0.105, 0.11 , 0.115, 0.12 , 0.125, 0.13 , 0.135, 0.14 ,
       0.145, 0.15 , 0.155, 0.16 , 0.165, 0.17 , 0.175, 0.18 , 0.185,
       0.19 , 0.195, 0.2  , 0.205, 0.21 , 0.215, 0.22 , 0.225, 0.23 ,
       0.235, 0.24 , 0.245, 0.25 ])

In [5]:
sampling_rate = 20 #Hz
freq_list = np.linspace(0.1, 0.4, 61)[:31]

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int)

A = 1
samples = 50
repeat = 200

interval = 50e-3
exposure_time = 500e-6 #500 us

measurement = 'DI'

In [8]:
############
# NO NOISE #
############

@email_notify('hcnzj@qq.com')
def main():
    with EasyDcam() as dcam, EasyALP4() as alp:
        for i, picture_time in enumerate(pic_time):
            print(f'({i + 1}): Current sensor temperature is {dcam.ez_temperature()}')
            if dcam.ez_temperature() >= -30:
                raise RuntimeError("qCMOS's temperature is too high.")

            ground_truth = np.round(freq_list[i], 5)

            alp.ez_load_seq([alp.ez_single_pixel(0), alp.ez_single_pixel(A)], picture_time)
            dcam.ez_exposure_time(exposure_time)
            dcam.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                dcam.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                dcam.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                dcam.buf_alloc(samples)
                dcam.cap_snapshot()

                alp.Run()
                sleep(1e-6)
                dcam.cap_firetrigger()

                dcam.ez_wait_capture()

                dcam.cap_stop()
                alp.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = dcam.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                dcam.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)

            raw = np.array(raw)
            timestamp = np.array(timestamp)

            if not os.path.exists(f'__raw__/{today}'):
                os.makedirs(f'__raw__/{today}')
            if not os.path.exists(f'__estimates__/{today}'):
                os.makedirs(f'__estimates__/{today}')

            metadata = MetaData(measurement, ground_truth, A*DMD.PIXEL_SIZE/2, timestamp)
            est = FrequencyEstimation(np.array(raw), metadata)
            est.savez(f'./__estimates__/{today}/{measurement.lower()}_{ground_truth}.npz')

            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_raw.npy', raw)
            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_timestamp.npy', timestamp)


if __name__ == '__main__':
    main()

qCMOS found, current sensor temperature is -36.0.
DMD found, resolution = 1024 x 768.
(1): Current sensor temperature is -36.0


  4%|▍         | 8/200 [00:59<23:56,  7.48s/it]


EasyALP4 exited
EasyDcam exited


KeyboardInterrupt: 

In [ ]:
raise RuntimeError('STOP HERE')

RuntimeError: STOP HERE